# Quest Hero: An RPG-Based Agentic AI Exercise

Welcome to **Quest Hero**, the capstone exercise in the AgenTekki agentic AI module!

You will build an LLM-powered agent that navigates an RPG grid world, interprets cryptic NPC dialogue, manages resources, and completes quests to collect **10 gold**.

## The Game
- **8x8 grid** with NPCs, shops, inns, monsters, forests, and treasures
- **Hero starts** at (7,0) with 3 hearts, 0 gold, empty inventory
- **Win condition**: Collect 10 gold and survive (hearts > 0)
- **Limited visibility**: You can only see adjacent cells
- **15 total gold** available through treasures, quests, and monster slaying

## Two Phases
1. **Phase 1 (Rule-Based)**: Write if/else logic to control the hero
2. **Phase 2 (LLM-Powered)**: Use an LLM to reason, plan, and interpret NPC clues

## What You Implement
| TODO | File | Description |
|------|------|-------------|
| 1 | `agent.py` | `think_rule_based()` — Rule-based decision logic |
| 2 | `oracle.py` | `build_npc_system_prompt()` + `ask_npc()` — NPC oracle |
| 3 | `agent.py` | `think_llm()` — LLM-powered decision function |

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("bmai_fs26_cx"):
        !git clone https://github.com/PLACEHOLDER_ORG/bmai_fs26_cx.git
    os.chdir("bmai_fs26_cx/coding-exercises/agentic_ai_hero")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from quest_hero import *
print("Quest Hero loaded successfully!")

---
## Exploring the Game

Before implementing your agent, let’s explore the game world and understand the tools available.

The hero interacts with the world through **9 tools**:
- `look()` — See adjacent cells (free)
- `move(direction)` — Move north/south/east/west
- `talk(question)` — Talk to NPCs, innkeepers, shopkeepers
- `pick_up()` — Pick up items in current cell
- `buy(item)` — Buy/craft at shops
- `use(item)` — Use consumables (Bread, Health Potion)
- `fight()` — Fight a monster (need correct weapon!)
- `rest()` — Rest at inn (costs 1 gold, +1 heart)
- `status()` — Check hero state

In [ ]:
# Create a game and explore manually
hero, world, tools = create_game()
tools.set_oracle(stub_oracle)

# See the full map (for exploration only — the agent sees limited visibility!)
from quest_hero.display import render_grid
print(render_grid(world, hero, show_all=True))

In [ ]:
# Try some tools manually
print("=== Look ===")
print(tools.execute("look", {}).message)

print("\n=== Move east (to Old Hermit) ===")
print(tools.execute("move", {"direction": "east"}).message)

print("\n=== Talk to the Hermit ===")
print(tools.execute("talk", {"question": "What dangers lie in the north?"}).message)

print("\n=== Status ===")
print(tools.execute("status", {}).message)

### Understanding the Tool Call Format

Your think functions must return a string in this format:
```
TOOL: tool_name(arg="value")
```

Examples:
```python
'TOOL: move(direction="north")'
'TOOL: pick_up()'
'TOOL: talk(question="Where can I find treasure?")'
'TOOL: buy(item="Sunblade")'
'TOOL: fight()'
```

The `parse_tool_call()` function extracts the tool name and arguments. If parsing fails, it defaults to `look()`.

In [ ]:
# Test the parser
from quest_hero.agent import parse_tool_call

print(parse_tool_call('TOOL: move(direction="north")'))
print(parse_tool_call('TOOL: pick_up()'))
print(parse_tool_call('I think I should move. TOOL: move(direction="east")'))
print(parse_tool_call('gibberish'))  # falls back to ('look', {})

---
## Phase 1: Rule-Based Agent

Implement `think_rule_based()` in `agent.py` — a function that decides the hero’s next action using if/else logic.

### Strategy Hints
- Use `hero.position` and `hero.visited` to navigate toward unexplored cells
- Check the current cell type (via `world.get_cell(row, col).cell_type`) to decide actions
- Priority: pick up treasures > talk to NPCs > explore new cells
- Navigate around mountains (cell_type == CellType.MOUNTAIN)
- The `history` list contains previous observations, actions, and results
- You need **10 gold** to win. Treasures give 1 gold each; quests give more.

### Available Information
```python
hero.position      # (row, col) tuple
hero.visited        # set of visited (row, col) tuples
hero.hearts         # current hearts (1-3)
hero.gold           # current gold
hero.inventory      # list of item name strings
hero.journal        # list of past events
hero.has_item(name) # check if hero has an item

world.get_cell(row, col)           # returns Cell with .cell_type, .items, .npc, etc.
world.get_adjacent(row, col)       # returns {"north": Cell, "south": Cell, ...}
world.is_passable(row, col)        # True if cell can be entered

CellType.TREASURE, CellType.NPC, CellType.SHOP, CellType.INN, etc.
```

In [ ]:
# Run Phase 1
result = play_rule_based(think_rule_based, max_turns=100, delay=0.1)
print(f"\nResult: {'Victory!' if result['won'] else 'Defeat...'} | Gold: {result['gold']} | Hearts: {result['hearts']} | Turns: {result['turns']}")

---
## Phase 2: LLM-Powered Agent

Now replace the if/else logic with an LLM that reasons about the game state, interprets NPC clues, and plans its actions.

You need to implement **two things**:
1. **NPC Oracle** (`oracle.py`): Make NPCs respond using the LLM
2. **Think function** (`agent.py`): Let the LLM decide the hero’s actions

### API Key Setup

Set your Gemini API key. In Colab, use the **Secrets** panel (key icon in the left sidebar) to add `GEMINI_API_KEY`.

In [ ]:
import os
from google import genai

# Option 1: Set your API key directly
# os.environ["GEMINI_API_KEY"] = "your-key-here"

# Option 2: In Colab, use Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

### TODO 2: NPC Oracle

Implement the NPC oracle so that NPCs respond to the hero using the LLM.

Each NPC has:
- `npc.name` — e.g., "Old Hermit"
- `npc.personality` — character description
- `npc.knowledge` — list of things the NPC knows (hints, not direct answers)
- `npc.style` — how the NPC speaks

The system prompt should make the NPC:
- Stay in character and never break the fourth wall
- Give hints based on their knowledge, not direct answers
- Be aware of what the hero is carrying
- Keep responses to 2-3 sentences

Reference the `ORACLE_TEMPLATE` in `oracle.py` for the structure.

In [ ]:
# TODO 2: Implement the NPC oracle

from quest_hero.game_world import NPC
from quest_hero.oracle import ORACLE_TEMPLATE

print("Template for reference:")
print(ORACLE_TEMPLATE)

In [ ]:
def build_npc_system_prompt(npc: NPC, hero: Hero) -> str:
    """
    Build a system prompt for an NPC.

    Use ORACLE_TEMPLATE.format(...) or build your own.
    """
    # ========================
    # YOUR CODE HERE
    # ========================
    pass


def ask_npc(npc: NPC, question: str, hero: Hero, client) -> str:
    """
    Ask an NPC a question using the Gemini API.

    1. Get the system prompt from build_npc_system_prompt()
    2. Call client.models.generate_content() with:
       - model="gemini-2.0-flash"
       - contents=question
       - config=genai.types.GenerateContentConfig(
             system_instruction=system_prompt,
             max_output_tokens=300,
         )
    3. Return response.text
    """
    # ========================
    # YOUR CODE HERE
    # ========================
    pass

In [ ]:
# Test your oracle with the Old Hermit
from quest_hero.game_world import NPC_CATALOG

hermit = NPC_CATALOG["old_hermit"]
test_hero = Hero()

response = ask_npc(hermit, "What dangers lie in the north?", test_hero, client)
print(f"Old Hermit says: {response}")
print()

# Test with the Forest Witch
witch = NPC_CATALOG["forest_witch"]
response = ask_npc(witch, "Where can I find Ember Ore?", test_hero, client)
print(f"Forest Witch says: {response}")

### TODO 3: LLM Think Function

Implement `think_llm()` — the LLM decides the hero’s next action each turn.

The function should:
1. **Build a system message** with:
   - Agent’s role ("You are a hero in an RPG, collect 10 gold to win")
   - Available tools (`TOOLS_DESCRIPTION`)
   - Instructions to output exactly one `TOOL: name(args)` call

2. **Build a user message** with:
   - Current hero status (`hero.status_text()`)
   - Recent history (last ~10 entries)
   - Key journal entries (past NPC conversations)

3. **Call the API** and return the response text

In [ ]:
# TODO 3: Implement the LLM-powered think function

from quest_hero.agent import TOOLS_DESCRIPTION


def think_llm(hero: Hero, world: GameWorld, history: list[dict]) -> str:
    """
    Decide the next action using an LLM.

    Must return a string containing: TOOL: tool_name(arg="value")

    Use client.models.generate_content() with:
    - model="gemini-2.0-flash"
    - contents=<your user message>
    - config=genai.types.GenerateContentConfig(
          system_instruction=<your system message>,
          max_output_tokens=500,
      )
    Return response.text
    """
    # ========================
    # YOUR CODE HERE
    # ========================
    pass

In [ ]:
# Run Phase 2
oracle_fn = lambda npc, q, h: ask_npc(npc, q, h, client)

result = play_with_llm(
    think_fn=lambda hero, world, history: think_llm(hero, world, history),
    oracle_fn=oracle_fn,
    max_turns=75,
    delay=0.3,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Defeat...'} | Gold: {result['gold']} | Hearts: {result['hearts']} | Turns: {result['turns']}")

---
## Reflection

Compare your Phase 1 (rule-based) and Phase 2 (LLM-powered) agents:

1. **Which collected more gold?** Why?
2. **How did each handle NPC clues?** The rule-based agent uses keyword matching; the LLM interprets natural language.
3. **What strategies did the LLM discover** that your rule-based agent couldn’t?
4. **What are the tradeoffs?** (cost, speed, reliability, flexibility)

### The Big Insight

> *The same perceive -> think -> act loop powers both agents. The loop didn’t change. The tools didn’t change. Only the `think` function evolved — from a simple heuristic to an LLM interpreting cryptic wizard dialogue. That’s the journey from a script to an agentic AI system.*